# Module 2: Recall Layer - Candidate Generation (MBA + ALS)

## Objective
Construct a high-recall candidate set (100-200 items per user) by combining:
1. **Organic Market Basket Analysis (MBA)** for familiarity signal.
2. **Implicit ALS Collaborative Filtering** for discovery signal.

## Methodological Principles
- Use **organic baskets** for MBA to reduce promotion-induced confounding.
- Use **retailer-revenue-weighted confidence** for ALS: $c_{ui} = 1 + \log_{10}(\text{Revenue\_Retailer}_{ui}+1)$.
- Normalize both recall signals to $[0,1]$ for downstream utility integration.
- Enforce initial uplift guardrail by excluding items purchased in the recent 4 weeks.

## Deliverables
- `association_rules.csv` with normalized MBA relevance.
- `user_item_matrix.npz` for ALS and downstream reproducibility.
- `candidate_set_module2.csv` with unioned, deduplicated, normalized relevance scores.

This notebook is written defensively: if processed parquet artifacts are unavailable (for example due to Git LFS pointers), it reconstructs required tables from raw CSV files.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz

from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
from implicit.als import AlternatingLeastSquares

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# ---- Reproducibility ----
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---- Runtime controls ----
SAMPLE_NROWS = 1_000_000      # Set to None for full corpus run
MAX_BASKETS_MBA = 150_000     # Cap baskets for tractable FP-Growth runtime
CANDIDATE_USERS_LIMIT = 500   # None for all users
TOP_ALS = 50
TOP_MBA = 50
SEED_ITEMS_K = 3
TOP_ALS_EXPORT = 100          # Task 2: Keep top-100 ALS scores per user
ALS_SCORE_BATCH_SIZE = 256

# ---- Project paths (robust to notebook launch location) ----
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "01_raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "02_processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("DATA_RAW exists:", DATA_RAW.exists())
print("DATA_PROCESSED exists:", DATA_PROCESSED.exists())

DATA_RAW exists: True
DATA_PROCESSED exists: True


In [17]:
from typing import Optional, Tuple


def _is_git_lfs_pointer(file_path: Path) -> bool:
    """Detect Git LFS pointer text files masquerading as artifacts."""
    if not file_path.exists() or file_path.stat().st_size == 0:
        return False
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            first_line = f.readline().strip().lower()
        return first_line.startswith("version https://git-lfs.github.com/spec")
    except UnicodeDecodeError:
        # Binary parquet typically raises decode error here, which is expected.
        return False


def _safe_read_parquet(file_path: Path) -> Optional[pd.DataFrame]:
    """Read parquet if valid; otherwise return None for fallback reconstruction."""
    if not file_path.exists():
        return None
    if _is_git_lfs_pointer(file_path):
        return None
    try:
        return pd.read_parquet(file_path)
    except Exception:
        return None


def _normalize_master_schema(df: pd.DataFrame, product_lookup: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """Standardize column naming and create required fields for Module 2."""
    rename_map = {}
    if "HOUSEHOLD_KEY" in df.columns and "household_key" not in df.columns:
        rename_map["HOUSEHOLD_KEY"] = "household_key"
    if "commodity_desc" in df.columns and "COMMODITY_DESC" not in df.columns:
        rename_map["commodity_desc"] = "COMMODITY_DESC"
    if rename_map:
        df = df.rename(columns=rename_map)

    if "COMMODITY_DESC" not in df.columns and product_lookup is not None and "PRODUCT_ID" in df.columns:
        df = df.merge(product_lookup, on="PRODUCT_ID", how="left")

    if "Revenue_Retailer" not in df.columns:
        if "SALES_VALUE" in df.columns:
            df["Revenue_Retailer"] = df["SALES_VALUE"].clip(lower=0)
        else:
            df["Revenue_Retailer"] = 0.0

    if "Is_Promoted_Item" not in df.columns:
        discount_cols = [c for c in ["RETAIL_DISC", "COUPON_DISC", "COUPON_MATCH_DISC"] if c in df.columns]
        if discount_cols:
            df["Is_Promoted_Item"] = (df[discount_cols].fillna(0) != 0).any(axis=1)
        else:
            df["Is_Promoted_Item"] = False

    required_cols = ["household_key", "BASKET_ID", "DAY", "COMMODITY_DESC", "Revenue_Retailer", "Is_Promoted_Item"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Master transaction data missing required columns: {missing}")

    commodity = df["COMMODITY_DESC"].fillna("UNKNOWN_COMMODITY").astype(str).str.strip()
    df["COMMODITY_DESC"] = np.where(commodity == "", "UNKNOWN_COMMODITY", commodity)
    df["Revenue_Retailer"] = pd.to_numeric(df["Revenue_Retailer"], errors="coerce").fillna(0).clip(lower=0)

    return df


def load_or_build_master_transactions(sample_nrows: Optional[int] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load processed master tables when available.
    Fallback: reconstruct from raw transaction + product tables.
    """
    all_path = DATA_PROCESSED / "master_transactions_all.parquet"
    organic_path = DATA_PROCESSED / "master_transactions_organic_only.parquet"

    product_lookup = pd.read_csv(DATA_RAW / "product.csv", usecols=["PRODUCT_ID", "COMMODITY_DESC"])

    all_df = _safe_read_parquet(all_path)
    organic_df = _safe_read_parquet(organic_path)

    if all_df is not None and organic_df is not None:
        all_df = _normalize_master_schema(all_df, product_lookup=product_lookup)
        organic_df = _normalize_master_schema(organic_df, product_lookup=product_lookup)
        print("Loaded processed parquet artifacts.")
        return all_df, organic_df

    print("Processed parquet artifacts unavailable; reconstructing from raw CSVs.")
    txn_cols = [
        "household_key", "BASKET_ID", "DAY", "PRODUCT_ID", "QUANTITY", "SALES_VALUE",
        "RETAIL_DISC", "COUPON_DISC", "COUPON_MATCH_DISC", "WEEK_NO"
    ]
    txn_df = pd.read_csv(DATA_RAW / "transaction_data.csv", usecols=txn_cols, nrows=sample_nrows)

    merged = txn_df.merge(product_lookup, on="PRODUCT_ID", how="left")
    merged["Is_Promoted_Item"] = (
        merged[["RETAIL_DISC", "COUPON_DISC", "COUPON_MATCH_DISC"]].fillna(0) != 0
    ).any(axis=1)

    # Conservative proxy for Module 2 confidence weighting.
    merged["Revenue_Retailer"] = merged["SALES_VALUE"].clip(lower=0)

    basket_is_promoted = merged.groupby("BASKET_ID")["Is_Promoted_Item"].transform("any")
    organic_df = merged.loc[~basket_is_promoted].copy()

    all_df = _normalize_master_schema(merged, product_lookup=product_lookup)
    organic_df = _normalize_master_schema(organic_df, product_lookup=product_lookup)

    # Persist only for full run to avoid accidentally storing sampled artifacts.
    if sample_nrows is None:
        all_df.to_parquet(all_path, index=False)
        organic_df.to_parquet(organic_path, index=False)
        print("Saved reconstructed master parquet artifacts.")

    return all_df, organic_df

In [18]:
tx_all, tx_org = load_or_build_master_transactions(sample_nrows=SAMPLE_NROWS)

print("all transactions shape:", tx_all.shape)
print("organic-only transactions shape:", tx_org.shape)
print("households:", tx_all["household_key"].nunique())
print("baskets:", tx_all["BASKET_ID"].nunique())
print("commodities:", tx_all["COMMODITY_DESC"].nunique())
print("day range:", int(tx_all["DAY"].min()), "->", int(tx_all["DAY"].max()))

promo_basket_share = tx_all.groupby("BASKET_ID")["Is_Promoted_Item"].any().mean()
print(f"share of promoted baskets: {promo_basket_share:.3f}")

display(tx_all.head(3))
display(tx_org.head(3))

Processed parquet artifacts unavailable; reconstructing from raw CSVs.
all transactions shape: (1000000, 13)
organic-only transactions shape: (38852, 13)
households: 2497
baskets: 108962
commodities: 302
day range: 1 -> 318
share of promoted baskets: 0.829


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,RETAIL_DISC,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC,COMMODITY_DESC,Is_Promoted_Item,Revenue_Retailer
0,2375,26984851472,1,1004906,1,1.39,-0.6,1,0.0,0.0,POTATOES,True,1.39
1,2375,26984851472,1,1033142,1,0.82,0.0,1,0.0,0.0,ONIONS,False,0.82
2,2375,26984851472,1,1036325,1,0.99,-0.3,1,0.0,0.0,VEGETABLES - ALL OTHERS,True,0.99


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,RETAIL_DISC,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC,COMMODITY_DESC,Is_Promoted_Item,Revenue_Retailer
21,1173,26984945254,1,824399,2,1.98,0.0,1,0.0,0.0,CANDY - PACKAGED,False,1.98
22,1173,26984945254,1,923972,1,0.67,0.0,1,0.0,0.0,CANDY - CHECKLANE,False,0.67
23,1173,26984945254,1,1131351,1,0.88,0.0,1,0.0,0.0,ELECTRICAL SUPPPLIES,False,0.88


In [19]:
from typing import Union


def normalize_lift_to_unit(lift_values: Union[np.ndarray, pd.Series]) -> np.ndarray:
    """
    Clamp and scale Lift into [0, 1] according to Module 2 specification:
    - Lift <= 1.0 -> 0.0
    - Lift >= 3.0 -> 1.0
    - Linear interpolation between 1.0 and 3.0
    """
    x = np.asarray(lift_values, dtype=float)
    out = np.where(x <= 1.0, 0.0, np.where(x >= 3.0, 1.0, (x - 1.0) / 2.0))
    return out


def build_mba_rules(
    organic_tx: pd.DataFrame,
    min_support: float = 0.001,
    max_len: int = 2,
    max_baskets: int = MAX_BASKETS_MBA,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fit FP-Growth on organic baskets and return long-format rule mappings."""
    filtered = organic_tx[~organic_tx["COMMODITY_DESC"].isin(["", "UNKNOWN_COMMODITY", "NO COMMODITY DESCRIPTION"])].copy()

    basket_items = (
        filtered.groupby("BASKET_ID")["COMMODITY_DESC"]
        .agg(lambda s: sorted(set(s.dropna().astype(str))))
    )
    basket_items = basket_items[basket_items.apply(len) >= 2]

    if len(basket_items) > max_baskets:
        basket_items = basket_items.sample(n=max_baskets, random_state=random_state)

    print(f"MBA baskets used: {len(basket_items):,}")

    encoder = TransactionEncoder()
    encoded = encoder.fit(basket_items.tolist()).transform(basket_items.tolist(), sparse=True)
    basket_ohe = pd.DataFrame.sparse.from_spmatrix(encoded, index=basket_items.index, columns=encoder.columns_)

    itemsets = fpgrowth(
        basket_ohe,
        min_support=min_support,
        use_colnames=True,
        max_len=max_len,
    )
    print(f"Frequent itemsets: {len(itemsets):,}")

    if itemsets.empty:
        empty = pd.DataFrame(columns=["antecedent_item", "consequent_item", "lift", "confidence", "support", "relevance_mba"])
        return empty, pd.DataFrame()

    rules = association_rules(itemsets, metric="lift", min_threshold=1.0)
    if rules.empty:
        empty = pd.DataFrame(columns=["antecedent_item", "consequent_item", "lift", "confidence", "support", "relevance_mba"])
        return empty, pd.DataFrame()

    rules = rules[(rules["consequents"].apply(len) == 1) & (rules["antecedents"].apply(len) >= 1)].copy()
    rules["consequent_item"] = rules["consequents"].apply(lambda s: next(iter(s)))
    rules["relevance_mba"] = normalize_lift_to_unit(rules["lift"])

    # Expand multi-antecedent rules into single-item trigger map for recency-seed lookup.
    rows = []
    for r in rules.itertuples(index=False):
        for ant in r.antecedents:
            rows.append(
                {
                    "antecedent_item": str(ant),
                    "consequent_item": str(r.consequent_item),
                    "lift": float(r.lift),
                    "confidence": float(r.confidence),
                    "support": float(r.support),
                    "relevance_mba": float(r.relevance_mba),
                }
            )

    rules_long = pd.DataFrame(rows)
    invalid_tokens = {"", "UNKNOWN_COMMODITY", "NO COMMODITY DESCRIPTION"}
    rules_long = rules_long[
        (~rules_long["antecedent_item"].isin(invalid_tokens))
        & (~rules_long["consequent_item"].isin(invalid_tokens))
        & (rules_long["antecedent_item"] != rules_long["consequent_item"])
    ].copy()

    rules_long = (
        rules_long
        .sort_values(["antecedent_item", "relevance_mba", "lift"], ascending=[True, False, False])
        .drop_duplicates(["antecedent_item", "consequent_item"])
        .reset_index(drop=True)
    )

    return rules_long, rules


mba_rules_long, mba_rules_raw = build_mba_rules(tx_org)
print("Expanded MBA rules:", len(mba_rules_long))
display(mba_rules_long.head(10))

MBA baskets used: 8,316
Frequent itemsets: 689
Expanded MBA rules: 588


,antecedent_item,consequent_item,lift,confidence,support,relevance_mba
0,ANALGESICS,COLD AND FLU,7.294737,0.114035,0.001563,1.000000
1,ANALGESICS,WATER - CARBONATED/FLVRD DRINK,1.723166,0.078947,0.001082,0.361583
2,ANALGESICS,CANDY - CHECKLANE,1.247933,0.096491,0.001323,0.123967
3,ANTACIDS,CIGARETTES,2.379777,0.180000,0.001082,0.689889
4,APPLES,CITRUS,6.211765,0.100840,0.001443,1.000000
5,APPLES,TROPICAL FRUIT,5.418685,0.243697,0.003487,1.000000
6,APPLES,SALAD BAR,2.169297,0.117647,0.001684,0.584648
7,APPLES,WATER - CARBONATED/FLVRD DRINK,1.650764,0.075630,0.001082,0.325382
8,BABY FOODS,DIAPERS & DISPOSABLES,10.171699,0.155340,0.001924,1.000000
9,BABY FOODS,BABY HBC,8.548715,0.087379,0.001082,1.000000


In [ ]:
def build_als_model(
    all_tx: pd.DataFrame,
    factors: int = 64,
    regularization: float = 0.05,
    iterations: int = 20,
    random_state: int = RANDOM_STATE,
):
    """Fit implicit ALS with confidence weighted by retailer revenue."""
    interactions = (
        all_tx.groupby(["household_key", "COMMODITY_DESC"], as_index=False)["Revenue_Retailer"]
        .sum()
    )

    users = np.sort(interactions["household_key"].unique())
    items = np.sort(interactions["COMMODITY_DESC"].astype(str).unique())

    user_to_idx = {int(u): int(i) for i, u in enumerate(users)}
    idx_to_user = {int(i): int(u) for i, u in enumerate(users)}
    item_to_idx = {str(it): int(j) for j, it in enumerate(items)}
    idx_to_item = {int(j): str(it) for j, it in enumerate(items)}

    u_idx = interactions["household_key"].map(user_to_idx).to_numpy(dtype=np.int32)
    i_idx = interactions["COMMODITY_DESC"].astype(str).map(item_to_idx).to_numpy(dtype=np.int32)

    confidence = 1.0 + np.log10(interactions["Revenue_Retailer"].to_numpy(dtype=float) + 1.0)
    confidence = np.clip(confidence, 1.0, None).astype(np.float32)

    user_item = csr_matrix((confidence, (u_idx, i_idx)), shape=(len(users), len(items)))

    model = AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=iterations,
        random_state=random_state,
    )
    model.fit(user_item)

    user_factors = np.asarray(model.user_factors, dtype=np.float32)
    item_factors = np.asarray(model.item_factors, dtype=np.float32)

    print("ALS matrix shape (users x items):", user_item.shape)
    print("ALS non-zero interactions:", user_item.nnz)
    print("ALS user factor shape:", user_factors.shape)
    print("ALS item factor shape:", item_factors.shape)

    return (
        model,
        user_item,
        user_to_idx,
        idx_to_user,
        item_to_idx,
        idx_to_item,
        users,
        items,
        user_factors,
        item_factors,
    )


def save_als_factors(
    users: np.ndarray,
    items: np.ndarray,
    user_factors: np.ndarray,
    item_factors: np.ndarray,
    user_to_idx: dict,
    item_to_idx: dict,
):
    """Persist ALS factors with deterministic mapping metadata for reproducible inference."""
    user_factor_path = DATA_PROCESSED / "user_factors.npz"
    item_factor_path = DATA_PROCESSED / "item_factors.npz"

    np.savez(
        user_factor_path,
        user_factors=user_factors,
        user_ids=np.asarray(users),
        user_id_to_index_json=np.array(json.dumps({str(k): int(v) for k, v in user_to_idx.items()})),
    )

    np.savez(
        item_factor_path,
        item_factors=item_factors,
        item_names=np.asarray(items, dtype=str),
        item_id_to_index_json=np.array(json.dumps({str(k): int(v) for k, v in item_to_idx.items()})),
    )

    print("Saved:")
    print(" -", user_factor_path)
    print(" -", item_factor_path)


(
    als_model,
    user_item_matrix,
    user_to_idx,
    idx_to_user,
    item_to_idx,
    idx_to_item,
    users,
    items,
    user_factors,
    item_factors,
) = build_als_model(tx_all)

save_als_factors(
    users=users,
    items=items,
    user_factors=user_factors,
    item_factors=item_factors,
    user_to_idx=user_to_idx,
    item_to_idx=item_to_idx,
)

  0%|          | 0/20 [00:00<?, ?it/s]

ALS matrix shape (users x items): (2497, 302)
ALS non-zero interactions: 198501


In [ ]:
def minmax_scale(values: np.ndarray) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr
    vmin, vmax = float(arr.min()), float(arr.max())
    if np.isclose(vmin, vmax):
        return np.ones_like(arr) if vmax > 0 else np.zeros_like(arr)
    return (arr - vmin) / (vmax - vmin)


def rowwise_minmax(arr_2d: np.ndarray) -> np.ndarray:
    mins = arr_2d.min(axis=1, keepdims=True)
    maxs = arr_2d.max(axis=1, keepdims=True)
    den = maxs - mins
    out = np.zeros_like(arr_2d, dtype=float)
    has_var = den.squeeze(1) > 0
    if np.any(has_var):
        out[has_var] = (arr_2d[has_var] - mins[has_var]) / den[has_var]
    no_var = ~has_var
    if np.any(no_var):
        out[no_var] = (maxs[no_var] > 0).astype(float)
    return out


def build_mba_lookup(rules_long: pd.DataFrame) -> dict:
    lookup = {}
    for ant, part in rules_long.groupby("antecedent_item"):
        seq = list(
            part.sort_values(["relevance_mba", "lift"], ascending=[False, False])[["consequent_item", "relevance_mba"]]
            .itertuples(index=False, name=None)
        )
        lookup[str(ant)] = [(str(item), float(score)) for item, score in seq]
    return lookup


def build_seed_items_table(all_tx: pd.DataFrame, k: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build k recency seed items per user: latest basket first, then recency backfill."""
    tx = all_tx.copy()

    latest_day = tx.groupby("household_key", as_index=False)["DAY"].max().rename(columns={"DAY": "latest_day"})
    tx_latest_day = tx.merge(latest_day, on="household_key", how="inner")
    tx_latest_day = tx_latest_day[tx_latest_day["DAY"] == tx_latest_day["latest_day"]].copy()

    latest_basket = tx_latest_day.groupby("household_key", as_index=False)["BASKET_ID"].max().rename(columns={"BASKET_ID": "latest_basket_id"})
    tx_latest_basket = tx_latest_day.merge(latest_basket, on="household_key", how="inner")
    tx_latest_basket = tx_latest_basket[tx_latest_basket["BASKET_ID"] == tx_latest_basket["latest_basket_id"]].copy()

    latest_items = (
        tx_latest_basket.sort_values(["household_key", "DAY", "BASKET_ID"], ascending=[True, False, False])
        .drop_duplicates(["household_key", "COMMODITY_DESC"])
        .groupby("household_key")["COMMODITY_DESC"]
        .apply(list)
        .reset_index(name="latest_items")
    )

    history_items = (
        tx.sort_values(["household_key", "DAY", "BASKET_ID"], ascending=[True, False, False])
        .drop_duplicates(["household_key", "COMMODITY_DESC"])
        .groupby("household_key")["COMMODITY_DESC"]
        .apply(list)
        .reset_index(name="history_items")
    )

    seeds = history_items.merge(latest_items, on="household_key", how="left")

    def _compose_seed(row):
        base = row["latest_items"] if isinstance(row["latest_items"], list) else []
        hist = row["history_items"] if isinstance(row["history_items"], list) else []
        out = []
        seen = set()
        for item in base + hist:
            if item not in seen:
                seen.add(item)
                out.append(str(item))
            if len(out) >= k:
                break
        return out

    seeds["seed_items_list"] = seeds.apply(_compose_seed, axis=1)
    seeds = seeds[seeds["seed_items_list"].str.len() > 0].copy()
    seeds["seed_items"] = seeds["seed_items_list"].apply(lambda x: " | ".join(x))

    seed_items_long = seeds[["household_key", "seed_items_list"]].explode("seed_items_list")
    seed_items_long = seed_items_long.rename(columns={"seed_items_list": "antecedent_item"})
    seed_items_long = seed_items_long.dropna().copy()

    seed_summary = seeds[["household_key", "seed_items"]].copy()
    return seed_items_long, seed_summary


def compute_als_scores_topk(
    users_arr: np.ndarray,
    items_arr: np.ndarray,
    user_factors_arr: np.ndarray,
    item_factors_arr: np.ndarray,
    user_indices: np.ndarray,
    top_k: int = 100,
    batch_size: int = 256,
) -> pd.DataFrame:
    """Compute ALS dot-product scores in batches and keep top-k per user."""
    items_obj = np.asarray(items_arr, dtype=object)
    n_items = item_factors_arr.shape[0]
    top_k = int(min(top_k, n_items))

    frames = []
    for start in range(0, len(user_indices), batch_size):
        batch_idx = user_indices[start:start + batch_size]
        batch_users = users_arr[batch_idx]
        batch_user_factors = user_factors_arr[batch_idx]

        scores = batch_user_factors @ item_factors_arr.T
        top_idx = np.argpartition(-scores, kth=top_k - 1, axis=1)[:, :top_k]
        top_scores = np.take_along_axis(scores, top_idx, axis=1)

        order = np.argsort(-top_scores, axis=1)
        top_idx = np.take_along_axis(top_idx, order, axis=1)
        top_scores = np.take_along_axis(top_scores, order, axis=1)

        top_scores_norm = rowwise_minmax(top_scores)

        frame = pd.DataFrame(
            {
                "household_key": np.repeat(batch_users, top_k),
                "COMMODITY_DESC": items_obj[top_idx.ravel()].astype(str),
                "relevance_als": top_scores_norm.ravel().astype(float),
            }
        )
        frames.append(frame)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["household_key", "COMMODITY_DESC", "relevance_als"])


# ---- Select user scope for this run ----
households = users.copy()
if CANDIDATE_USERS_LIMIT is not None:
    households = households[:CANDIDATE_USERS_LIMIT]

selected_user_indices = np.array([user_to_idx[int(h)] for h in households], dtype=int)

# ---- Task 2: ALS scores export (Top-100/user) ----
als_scores = compute_als_scores_topk(
    users_arr=users,
    items_arr=items,
    user_factors_arr=user_factors,
    item_factors_arr=item_factors,
    user_indices=selected_user_indices,
    top_k=TOP_ALS_EXPORT,
    batch_size=ALS_SCORE_BATCH_SIZE,
)

als_scores["relevance_als"] = als_scores["relevance_als"].clip(0, 1)
als_scores_path = DATA_PROCESSED / "als_scores.parquet"
als_scores.to_parquet(als_scores_path, index=False)

print("ALS top-100 score rows:", len(als_scores))
print("Saved:", als_scores_path)

# ---- MBA candidates from seed items (vectorized joins) ----
mba_lookup = build_mba_lookup(mba_rules_long)
seed_items_long, seed_summary = build_seed_items_table(tx_all[tx_all["household_key"].isin(households)], k=SEED_ITEMS_K)

if seed_items_long.empty:
    mba_candidates = pd.DataFrame(columns=["household_key", "COMMODITY_DESC", "relevance_mba"])
else:
    mba_candidates = seed_items_long.merge(
        mba_rules_long[["antecedent_item", "consequent_item", "relevance_mba"]],
        on="antecedent_item",
        how="inner",
    )
    mba_candidates = (
        mba_candidates.groupby(["household_key", "consequent_item"], as_index=False)["relevance_mba"]
        .max()
        .rename(columns={"consequent_item": "COMMODITY_DESC"})
    )
    mba_candidates = (
        mba_candidates.sort_values(["household_key", "relevance_mba"], ascending=[True, False])
        .groupby("household_key", as_index=False)
        .head(TOP_MBA)
        .reset_index(drop=True)
    )

# ---- ALS candidates for union (top-50/user from top-100 export) ----
als_candidates = (
    als_scores.sort_values(["household_key", "relevance_als"], ascending=[True, False])
    .groupby("household_key", as_index=False)
    .head(TOP_ALS)
    .reset_index(drop=True)
)

# ---- Recency window + snapshot ----
max_day = int(tx_all["DAY"].max())
recent_cutoff_day = max_day - 28
snapshot_week = int(tx_all["WEEK_NO"].max()) if "WEEK_NO" in tx_all.columns else int(np.ceil(max_day / 7))

recent_pairs = (
    tx_all[(tx_all["DAY"] >= recent_cutoff_day) & (tx_all["household_key"].isin(households))][["household_key", "COMMODITY_DESC"]]
    .drop_duplicates()
    .assign(was_recently_purchased=True)
)

print("users selected:", len(households))
print("ALS candidates (top-50/user):", len(als_candidates))
print("MBA candidates (top-50/user):", len(mba_candidates))

In [ ]:
# ---- Candidate union ----
unified_candidates = als_candidates.merge(
    mba_candidates,
    on=["household_key", "COMMODITY_DESC"],
    how="outer",
)

unified_candidates["relevance_als"] = unified_candidates["relevance_als"].fillna(0.0)
unified_candidates["relevance_mba"] = unified_candidates["relevance_mba"].fillna(0.0)
unified_candidates["Relevance"] = unified_candidates[["relevance_als", "relevance_mba"]].max(axis=1)

unified_candidates["source_detail"] = np.select(
    [
        (unified_candidates["relevance_als"] > 0) & (unified_candidates["relevance_mba"] > 0),
        (unified_candidates["relevance_als"] > 0) & (unified_candidates["relevance_mba"] == 0),
        (unified_candidates["relevance_als"] == 0) & (unified_candidates["relevance_mba"] > 0),
    ],
    ["BOTH", "ALS", "MBA"],
    default="UNKNOWN",
)
unified_candidates["source"] = unified_candidates["source_detail"].str.lower()
unified_candidates["snapshot_week"] = snapshot_week

unified_candidates = unified_candidates.merge(seed_summary, on="household_key", how="left")

# ---- Task 4: Filter traceability ----
unified_candidates = unified_candidates.merge(
    recent_pairs,
    on=["household_key", "COMMODITY_DESC"],
    how="left",
)
unified_candidates["was_recently_purchased"] = unified_candidates["was_recently_purchased"].fillna(False)

filtered_items_log = (
    unified_candidates[unified_candidates["was_recently_purchased"]][["household_key", "COMMODITY_DESC"]]
    .drop_duplicates()
    .assign(filter_reason="recent_purchase", reference_week=snapshot_week)
)

candidate_set = unified_candidates[~unified_candidates["was_recently_purchased"]].copy()

# Ensure strict [0,1] compliance
for col in ["relevance_als", "relevance_mba", "Relevance"]:
    candidate_set[col] = candidate_set[col].clip(0, 1)

# ---- Persist Module 2 outputs ----
mba_rules_long.to_csv(DATA_PROCESSED / "association_rules.csv", index=False)
save_npz(DATA_PROCESSED / "user_item_matrix.npz", user_item_matrix)

candidate_set = candidate_set[
    [
        "household_key",
        "COMMODITY_DESC",
        "seed_items",
        "relevance_als",
        "relevance_mba",
        "Relevance",
        "source",
        "source_detail",
        "snapshot_week",
        "was_recently_purchased",
    ]
]

candidate_set.to_csv(DATA_PROCESSED / "candidate_set_module2.csv", index=False)
filtered_items_log.to_csv(DATA_PROCESSED / "filtered_items_log.csv", index=False)

print("Saved:")
print(" -", DATA_PROCESSED / "association_rules.csv")
print(" -", DATA_PROCESSED / "user_item_matrix.npz")
print(" -", DATA_PROCESSED / "user_factors.npz")
print(" -", DATA_PROCESSED / "item_factors.npz")
print(" -", DATA_PROCESSED / "als_scores.parquet")
print(" -", DATA_PROCESSED / "candidate_set_module2.csv")
print(" -", DATA_PROCESSED / "filtered_items_log.csv")

print("\nusers generated:", candidate_set["household_key"].nunique())
print("candidate rows:", len(candidate_set))
print("avg candidates per user:", round(candidate_set.groupby("household_key").size().mean(), 2))
print("source mix:")
print(candidate_set["source_detail"].value_counts(normalize=True).round(3))

display(candidate_set.head(20))
display(filtered_items_log.head(20))

users generated: 500
candidate rows: 30577
avg candidates per user: 61.15
source mix:
source
als     0.730
mba     0.199
both    0.071
Name: proportion, dtype: float64


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source
0,1,FROZEN PIZZA,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,1.000000,0.000000,1.000000,als
1,1,YOGURT,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,0.341257,1.000000,1.000000,both
2,1,GRAPES,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,0.000000,1.000000,1.000000,mba
3,1,TOMATOES,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,0.000000,1.000000,1.000000,mba
4,1,STONE FRUIT,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,0.000000,1.000000,1.000000,mba
5,1,VALUE ADDED FRUIT,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,0.264136,1.000000,1.000000,both
6,1,BERRIES,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,0.000000,1.000000,1.000000,mba
7,1,PAPER TOWELS,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,0.965025,0.000000,0.965025,als
8,1,VEGETABLES - ALL OTHERS,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,0.000000,0.919274,0.919274,mba
9,1,HOT CEREAL,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,0.800046,0.000000,0.800046,als


Saved:
 - /Users/minhduc/Documents/Zoom/2026-04-01 19.04.18 Cuộc họp Zoom của Minh Đức/Power BI Projects updated/Data_Driven_Marketing_complete-journey/data/02_processed/association_rules.csv
 - /Users/minhduc/Documents/Zoom/2026-04-01 19.04.18 Cuộc họp Zoom của Minh Đức/Power BI Projects updated/Data_Driven_Marketing_complete-journey/data/02_processed/user_item_matrix.npz
 - /Users/minhduc/Documents/Zoom/2026-04-01 19.04.18 Cuộc họp Zoom của Minh Đức/Power BI Projects updated/Data_Driven_Marketing_complete-journey/data/02_processed/candidate_set_module2.csv


In [ ]:
if not candidate_set.empty:
    # Task 5.1 Average candidates per user
    avg_candidates = candidate_set.groupby("household_key")["COMMODITY_DESC"].nunique().mean()
    print("1) Avg candidates per user:", round(float(avg_candidates), 2), "(target: 100-200)")

    # Task 5.2 Source composition
    source_pct = candidate_set["source_detail"].value_counts(normalize=True).mul(100).round(2)
    print("\n2) Source composition (%):")
    for key in ["ALS", "MBA", "BOTH"]:
        print(f" - {key}: {source_pct.get(key, 0.0)}%")

    # Task 5.3 Recency removals
    removed_count = len(filtered_items_log)
    pre_filter_count = len(unified_candidates)
    removed_pct = (removed_count / pre_filter_count * 100.0) if pre_filter_count > 0 else 0.0
    print("\n3) Removed due to recency filter:", round(removed_pct, 2), "%")

    # Task 5.4 Verify no leakage of recent items
    leak_check = candidate_set.merge(
        recent_pairs[["household_key", "COMMODITY_DESC"]],
        on=["household_key", "COMMODITY_DESC"],
        how="inner",
    )
    print("\n4) Leakage check (must be 0):", len(leak_check))

    # Additional QA diagnostics
    print("\nCandidate count per user quantiles:")
    coverage = candidate_set.groupby("household_key")["COMMODITY_DESC"].nunique()
    print(coverage.quantile([0.0, 0.25, 0.5, 0.75, 1.0]).round(2))

    for c in ["relevance_als", "relevance_mba", "Relevance"]:
        low_ok = bool((candidate_set[c] >= 0.0).all())
        high_ok = bool((candidate_set[c] <= 1.0).all())
        print(f"{c}: within [0,1] ->", low_ok and high_ok)

    print("\nSample top-5 recommendations for 3 users:")
    for hh in candidate_set["household_key"].drop_duplicates().head(3):
        print(f"\nHousehold {hh}")
        display(candidate_set[candidate_set["household_key"] == hh].sort_values("Relevance", ascending=False).head(5))

candidate count per user quantiles:
0.00    50.0
0.25    54.0
0.50    61.0
0.75    67.0
1.00    87.0
Name: COMMODITY_DESC, dtype: float64
relevance_als: within [0,1] -> True
relevance_mba: within [0,1] -> True
Relevance: within [0,1] -> True

Sample top-5 recommendations for 3 users:

Household 1


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source
0,1,FROZEN PIZZA,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,1.000000,0.0,1.0,als
1,1,YOGURT,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,0.341257,1.0,1.0,both
2,1,GRAPES,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,0.000000,1.0,1.0,mba
3,1,TOMATOES,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,0.000000,1.0,1.0,mba
4,1,STONE FRUIT,TROPICAL FRUIT | CANDY - PACKAGED | DRY MIX DE...,0.000000,1.0,1.0,mba



Household 2


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source
64,2,FROZEN PIZZA,VALUE ADDED VEGETABLES | SOFT DRINKS | CIGARETTES,1.000000,0.496923,1.000000,both
65,2,TOBACCO OTHER,VALUE ADDED VEGETABLES | SOFT DRINKS | CIGARETTES,0.000000,1.000000,1.000000,mba
66,2,FUEL,VALUE ADDED VEGETABLES | SOFT DRINKS | CIGARETTES,0.000000,1.000000,1.000000,mba
67,2,DRY BN/VEG/POTATO/RICE,VALUE ADDED VEGETABLES | SOFT DRINKS | CIGARETTES,0.998038,0.000000,0.998038,als
68,2,VALUE ADDED FRUIT,VALUE ADDED VEGETABLES | SOFT DRINKS | CIGARETTES,0.949108,0.000000,0.949108,als



Household 3


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source
139,3,EGGS,BEEF | REFRGRATD DOUGH PRODUCTS | VEGETABLES -...,1.000000,0.314655,1.0,both
140,3,PEPPERS-ALL,BEEF | REFRGRATD DOUGH PRODUCTS | VEGETABLES -...,0.260349,1.000000,1.0,both
141,3,HISPANIC,BEEF | REFRGRATD DOUGH PRODUCTS | VEGETABLES -...,0.000000,1.000000,1.0,mba
142,3,DRY BN/VEG/POTATO/RICE,BEEF | REFRGRATD DOUGH PRODUCTS | VEGETABLES -...,0.000000,1.000000,1.0,mba
143,3,POTATOES,BEEF | REFRGRATD DOUGH PRODUCTS | VEGETABLES -...,0.000000,1.000000,1.0,mba


## Handoff to Module 3 (Utility Ranking)

This notebook now exports a reproducible and explainable Module 2 artifact set:

- `association_rules.csv`: normalized MBA signal (`relevance_mba`).
- `als_scores.parquet`: top-100 ALS dot-product scores per user, normalized to `[0,1]`.
- `user_factors.npz`: ALS user latent factors + `user_id_to_index` mapping metadata.
- `item_factors.npz`: ALS item latent factors + `item_id_to_index` mapping metadata.
- `candidate_set_module2.csv`: recall-union candidates with source traceability fields.
- `filtered_items_log.csv`: explicit audit log of recency-based removals.
- `user_item_matrix.npz`: sparse confidence matrix cache.

Candidate set columns include:
- `relevance_als`
- `relevance_mba`
- `Relevance = max(relevance_als, relevance_mba)`
- `source_detail` in `{ALS, MBA, BOTH}`
- `snapshot_week`
- `was_recently_purchased`

Module 3 can directly consume `candidate_set_module2.csv` and combine with margin, uplift, and context features.

$$
U(i,u)=w_1\cdot Relevance(i,u)+w_2\cdot Uplift(i,u)+w_3\cdot Margin(i)+w_4\cdot Context(i,u)
$$

with baseline weights:
- $w_1=0.40$
- $w_2=0.25$
- $w_3=0.20$
- $w_4=0.15$.